In [1]:
import pyspark.sql.functions as F
from pyspark.sql.types import * 

In [2]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog= dbutils.widgets.get("catalog")

/mnt/c/Users/ishfa/OneDrive/Desktop/Projects/dab_project/.venv_dbc/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:71: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(

KeyboardInterrupt



In [3]:
landing_path = "/Volumes/citibikes_dev/00_landing/operational_data/rides/*"

In [4]:
schema = StructType(
    [StructField("rider_d", StringType(), True),
     StructField("rideable_type", StringType(),True),
     StructField("started_at", TimestampType(), True),
     StructField("ended_at",TimestampType(),True),
     StructField("start_station_name", StringType(),True),
     StructField('start_station_id', StringType(),True),
     StructField("end_station_name", StringType(), True),
     StructField("end_station_id", StringType(),True),
     StructField("start_lat", DecimalType(), True),
     StructField("start_lng", DecimalType(),True),
     StructField("end_lat", DecimalType(), True),
     StructField("end_lng", DecimalType(),True),
     StructField("member_casual", StringType(),True)
]
)

In [5]:
df = spark.read.format('csv').load(landing_path, schema=schema, header=True)

In [6]:
df.head(5)

[Row(rider_d='29DAF43DD84B4B7A', rideable_type='electric_bike', started_at=datetime.datetime(2025, 3, 20, 18, 58, 31, 217000), ended_at=datetime.datetime(2025, 3, 20, 19, 0, 46, 466000), start_station_name='6 St & Grand St', start_station_id='HB302', end_station_name='Mama Johnson Field - 4 St & Jackson St', end_station_id='HB404', start_lat=Decimal('41'), start_lng=Decimal('-74'), end_lat=Decimal('41'), end_lng=Decimal('-74'), member_casual='member'),
 Row(rider_d='B11B4220F7195025', rideable_type='electric_bike', started_at=datetime.datetime(2025, 3, 29, 11, 1, 25, 124000), ended_at=datetime.datetime(2025, 3, 29, 11, 11, 9, 383000), start_station_name='Heights Elevator', start_station_id='JC059', end_station_name='Jersey & 3rd', end_station_id='JC074', start_lat=Decimal('41'), start_lng=Decimal('-74'), end_lat=Decimal('41'), end_lng=Decimal('-74'), member_casual='member'),
 Row(rider_d='18D5B30305F602B9', rideable_type='electric_bike', started_at=datetime.datetime(2025, 3, 1, 16, 5, 

In [ ]:
column_map =F.create_map(
    F.lit("pipeline_id"), F.lit(pipeline_id),
    F.lit("run_id"), F.lit(run_id),
    F.lit("task_id"), F.lit(task_id),
    F.lit("processed_timestamp"), F.lit(processed_timestamp))


In [5]:
df = df.withColumn("metadata", column_map)

In [ ]:
df.writeTo(f"{catalog}.01_bronze.jc_citibikes").createOrReplace()
